In [1]:
import matplotlib.pyplot as plt
import sys
import os
import importlib

# Add the parent directory to the path to import src as a package
sys.path.insert(0, os.path.abspath('..'))
from src import dataloader

importlib.reload(dataloader)
from src import utils
from src import export

importlib.reload(export)
from src.export import export_to_xarray



%matplotlib widget 
plot_flag = False

In [2]:
dyad_id = "W_030"
lowcut=1.0
highcut=40.0
eeg_filter_type = 'iir' # choose 'fir' or 'iir' for EEG filtering
q=8  # decimation factor
multimodal_data = dataloader.create_multimodal_data(data_base_path = "../data", 
                                                    dyad_id = dyad_id, 
                                                    load_eeg=True, 
                                                    load_et=True, 
                                                    load_meta = True, 
                                                    lowcut=lowcut, 
                                                    highcut=highcut, 
                                                    eeg_filter_type=eeg_filter_type, 
                                                    interpolate_et_during_blinks_threshold=0.3,
                                                    median_filter_size=64,
                                                    low_pass_et_order=351,
                                                    et_pos_cutoff=128,
                                                    et_pupil_cutoff=4,
                                                    pupil_model_confidence=0.9,
                                                    decimate_factor=q,
                                                    plot_flag=plot_flag)

Detected events: [{'name': 'Brave', 'start': 387.806640625, 'duration': 59.3310546875}, {'name': 'Peppa', 'start': 248.5107421875, 'duration': 59.6328125}, {'name': 'Incredibles', 'start': 318.3603515625, 'duration': 59.212890625}, {'name': 'Talk_1', 'start': 594.4892578125, 'duration': 181.0556640625}, {'name': 'Talk_2', 'start': 836.7275390625, 'duration': 181.056640625}]
Applying iir filters to EEG data.
Reseting the EEG time to the start of Peppa
ET time range: 241.59s to 461.89s
Events from ET annotations:
[None 'Peppa' 'Incredibles' 'Brave']
Reseting the ET time to the start of Peppa
Processing member: ch, blink column: ET_ch_blinks
Processing member: cg, blink column: ET_cg_blinks
Column ET_ch_blinks contains NaN values, applying forward fill before decimation.
Column ET_cg_blinks contains NaN values, applying forward fill before decimation.
Event Peppa start times are consistent within 0.0 seconds.
Event Incredibles differ in start times by: abs(0.0078125) seconds.
Event Brave 

In [3]:
multimodal_data.eeg_filtration.notch['Q']


30

In [ ]:
 # seconds
#selected_event =  'Peppa' #'Incredibles' 'Peppa' 'Brave' 'Talk_1' 'Talk_2'
#member = 'ch' #'cg'

#selected_modality = 'EEG' # choose 'EEG', 'ECG', 'ET' , 'IBI' or 'diode' for modality to export to xarray (diode is the reference for checking the correctnes of event slicing)
#selected_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'M1', 'T3', 'C3', 'Cz', 'C4', 'T4', 'M2', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
#selected_modality = 'ET'
#selected_channels = ['x', 'y', 'pupil', 'blinks']

#selected_modality = 'ECG'
#selected_channels = ['ECG']

#selected_modality = 'IBI'
#selected_channels = ['IBI']
#selected_modality = 'diode'
#selected_channels = ['diode']
time_margin = 10
selected_channels = {'EEG': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'M1', 'T3', 'C3', 'Cz', 'C4', 'T4', 'M2', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'],
                     'ET':['x', 'y', 'pupil', 'blinks'],
                     'ECG':['ECG'],
                     'IBI':['IBI']}
selected_modalities = multimodal_data.modalities
members = {'ch':'child', 'cg':'caregiver'}

root_path = "../data/UNIWAW_imported"
path_dyad = os.path.join(root_path, str(multimodal_data.id))
if not os.path.exists(path_dyad):
    os.makedirs(path_dyad)

for modality in selected_modalities:
    path_modality = os.path.join(path_dyad, modality)
    if not os.path.exists(path_modality):
        os.makedirs(path_modality)
    for who, member in members.items():
        path_member = os.path.join(path_modality, member)
        if not os.path.exists(path_member):
            os.makedirs(path_member)
        for event in multimodal_data.events.keys():    
            data_xr = export_to_xarray(multimodal_data = multimodal_data, 
                                   selected_event=event, 
                                   selected_channels=selected_channels.get(modality), 
                                   selected_modality=modality, 
                                   member=who, 
                                   time_margin=time_margin)
            file_path = os.path.join(path_member, f'{multimodal_data.id}_{modality}_{who}_{event}.nc')
            data_xr.to_netcdf(file_path, engine='netcdf4')






Event 'Peppa' starts at 0.00s and ends at 59.63s
Selected time range with ±10s margin: -10.00s to 69.63s
Event 'Incredibles' starts at 69.86s and ends at 129.06s
Selected time range with ±10s margin: 59.86s to 139.06s
Event 'Brave' starts at 139.30s and ends at 198.62s
Selected time range with ±10s margin: 129.30s to 208.62s
Event 'Talk_1' starts at 345.98s and ends at 527.03s
Selected time range with ±10s margin: 335.98s to 537.03s
Event 'Talk_2' starts at 588.22s and ends at 769.27s
Selected time range with ±10s margin: 578.22s to 779.27s
Event 'Peppa' starts at 0.00s and ends at 59.63s
Selected time range with ±10s margin: -10.00s to 69.63s
Event 'Incredibles' starts at 69.86s and ends at 129.06s
Selected time range with ±10s margin: 59.86s to 139.06s
Event 'Brave' starts at 139.30s and ends at 198.62s
Selected time range with ±10s margin: 129.30s to 208.62s
Event 'Talk_1' starts at 345.98s and ends at 527.03s
Selected time range with ±10s margin: 335.98s to 537.03s
Event 'Talk_2' s